# PRODUCTS.CSV

In [29]:
# ==========================================================
# PRODUCTS.CSV AUDIT
# ==========================================================
import duckdb

con = duckdb.connect()
path = "../../source_database/seed_data/products.csv"

In [2]:
# 1. SCHEMA AUDIT
con.sql(f"DESCRIBE SELECT * FROM read_csv_auto('{path}')").show()

┌─────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│   column_name   │ column_type │  null   │   key   │ default │  extra  │
│     varchar     │   varchar   │ varchar │ varchar │ varchar │ varchar │
├─────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ product_id      │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ sku             │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ product_name    │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ brand           │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ category_l1     │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ category_l2     │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ category_l3     │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ unit_of_measure │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ pack_size       │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ is_own_brand    │ BOOLEAN     │ YES 

In [3]:
con.sql(f"SELECT * FROM read_csv_auto('{path}') LIMIT 5").show()

┌────────────┬──────────────┬──────────────────────┬───┬───────────┬─────────────────────┬─────────────────────┐
│ product_id │     sku      │     product_name     │ … │ is_active │     created_at      │     updated_at      │
│   int64    │   varchar    │       varchar        │   │  boolean  │      timestamp      │      timestamp      │
├────────────┼──────────────┼──────────────────────┼───┼───────────┼─────────────────────┼─────────────────────┤
│          1 │ NP-SKU-00001 │ Kelloggs Fruit 6pk   │ … │ true      │ 2020-01-01 00:00:00 │ 2026-01-01 00:00:00 │
│          2 │ NP-SKU-00002 │ Warburtons Milk 1kg  │ … │ true      │ 2020-01-01 00:00:00 │ 2026-01-01 00:00:00 │
│          3 │ NP-SKU-00003 │ Walkers Cheese 1kg   │ … │ true      │ 2020-01-01 00:00:00 │ 2026-01-01 00:00:00 │
│          4 │ NP-SKU-00004 │ Kelloggs Yoghurt 5…  │ … │ true      │ 2020-01-01 00:00:00 │ 2026-01-01 00:00:00 │
│          5 │ NP-SKU-00005 │ Nestle Beef 6pk      │ … │ true      │ 2020-01-01 00:00:00 │ 2026-

In [4]:
# 2. COMPLETENESS AUDIT
total_rows = con.sql(f"SELECT COUNT(*) FROM read_csv_auto('{path}')").fetchone()[0]
print(f"Total rows: {total_rows}")

cols = con.sql(f"DESCRIBE SELECT * FROM read_csv_auto('{path}')").df()['column_name'].tolist()
for col in cols:
    null_count = con.sql(f"SELECT COUNT(*) - COUNT({col}) FROM read_csv_auto('{path}')").fetchone()[0]
    pct = round(null_count / total_rows * 100, 2)
    print(f"{col}: {null_count} nulls ({pct}%)")

Total rows: 150
product_id: 0 nulls (0.0%)
sku: 0 nulls (0.0%)
product_name: 0 nulls (0.0%)
brand: 0 nulls (0.0%)
category_l1: 0 nulls (0.0%)
category_l2: 0 nulls (0.0%)
category_l3: 0 nulls (0.0%)
unit_of_measure: 0 nulls (0.0%)
pack_size: 0 nulls (0.0%)
is_own_brand: 0 nulls (0.0%)
supplier_id: 0 nulls (0.0%)
rrp: 0 nulls (0.0%)
cost_price: 0 nulls (0.0%)
vat_rate: 0 nulls (0.0%)
is_active: 0 nulls (0.0%)
created_at: 0 nulls (0.0%)
updated_at: 0 nulls (0.0%)


In [5]:
# 3. UNIQUENESS AUDIT — product_id (PK) and sku should both be unique
for col in ["product_id", "sku"]:
    dupes = con.sql(f"""
        SELECT {col}, COUNT(*) AS row_count FROM read_csv_auto('{path}')
        GROUP BY {col} HAVING COUNT(*) > 1
    """).df()
    print(f"{col}: {len(dupes)} duplicated values")

product_id: 0 duplicated values
sku: 0 duplicated values


In [6]:
# 4. REFERENTIAL INTEGRITY AUDIT — supplier_id must exist in suppliers.csv
orphan_suppliers = con.sql(f"""
    SELECT COUNT(*) FROM read_csv_auto('{path}') p
    LEFT JOIN read_csv_auto('../../source_database/seed_data/suppliers.csv') s ON p.supplier_id = s.supplier_id
    WHERE s.supplier_id IS NULL
""").fetchone()[0]
print(f"Orphaned supplier_id rows: {orphan_suppliers}")

Orphaned supplier_id rows: 0


In [7]:
# 5. VALIDITY / BUSINESS RULE AUDIT

# selling below cost would be a real pricing error
selling_at_loss = con.sql(f"""
    SELECT COUNT(*) FROM read_csv_auto('{path}') WHERE cost_price > rrp
""").fetchone()[0]
print(f"Products where cost_price > rrp (selling at a loss): {selling_at_loss}")

# UK VAT rule: Food should be 0% (zero-rated), everything else should be 20%
# Test the rule, don't assume it
vat_violations = con.sql(f"""
    SELECT COUNT(*) FROM read_csv_auto('{path}')
    WHERE (category_l1 = 'Food' AND vat_rate != 0.0)
       OR (category_l1 != 'Food' AND vat_rate != 0.2)
""").fetchone()[0]
print(f"Rows breaking the Food=0% VAT / non-Food=20% VAT pattern: {vat_violations}")

# pack_size, rrp, cost_price should all be positive
for col in ["pack_size", "rrp", "cost_price"]:
    bad = con.sql(f"SELECT COUNT(*) FROM read_csv_auto('{path}') WHERE {col} <= 0").fetchone()[0]
    print(f"{col} <= 0: {bad}")

Products where cost_price > rrp (selling at a loss): 0
Rows breaking the Food=0% VAT / non-Food=20% VAT pattern: 0
pack_size <= 0: 0
rrp <= 0: 0
cost_price <= 0: 0


In [8]:
# 6. BRAND vs PRODUCT_NAME CONSISTENCY CHECK
# Does the brand mentioned in product_name actually match the brand column?
# (I already spot-checked this — expect a high mismatch rate, this confirms the real number)
mismatch_df = con.sql(f"""
    SELECT COUNT(*) AS mismatches
    FROM read_csv_auto('{path}')
    WHERE NOT REGEXP_MATCHES(LOWER(product_name), LOWER(SPLIT_PART(brand, ' ', 1)))
""").df()
total = con.sql(f"SELECT COUNT(*) FROM read_csv_auto('{path}')").fetchone()[0]
mismatch_count = mismatch_df['mismatches'][0]
print(f"product_name doesn't mention its own brand: {mismatch_count} / {total} ({round(mismatch_count/total*100,1)}%)")

product_name doesn't mention its own brand: 137 / 150 (91.3%)


In [9]:
# 7. DISTRIBUTION / OUTLIER AUDIT — margin (rrp - cost_price) via IQR
con.sql(f"""
    SELECT MIN(rrp) AS min_rrp, MAX(rrp) AS max_rrp, AVG(rrp) AS avg_rrp,
           MIN(cost_price) AS min_cost, MAX(cost_price) AS max_cost
    FROM read_csv_auto('{path}')
""").show()

iqr = con.sql(f"""
    SELECT QUANTILE_CONT(rrp - cost_price, 0.25) AS q1, QUANTILE_CONT(rrp - cost_price, 0.75) AS q3
    FROM read_csv_auto('{path}')
""").df()
q1, q3 = iqr['q1'][0], iqr['q3'][0]
iqr_range = q3 - q1
lower, upper = q1 - 1.5*iqr_range, q3 + 1.5*iqr_range
outliers = con.sql(f"""
    SELECT product_id, product_name, rrp, cost_price, (rrp - cost_price) AS margin
    FROM read_csv_auto('{path}')
    WHERE (rrp - cost_price) < {lower} OR (rrp - cost_price) > {upper}
""").df()
print(f"Margin outliers (unusually high/low profit): {len(outliers)}")
outliers

┌─────────┬─────────┬──────────────────┬──────────┬──────────┐
│ min_rrp │ max_rrp │     avg_rrp      │ min_cost │ max_cost │
│ double  │ double  │      double      │  double  │  double  │
├─────────┼─────────┼──────────────────┼──────────┼──────────┤
│    0.53 │   11.98 │ 6.13106666666667 │     0.24 │     7.44 │
└─────────┴─────────┴──────────────────┴──────────┴──────────┘

Margin outliers (unusually high/low profit): 0


,product_id,product_name,rrp,cost_price,margin


In [10]:
# 8. FRESHNESS AUDIT
con.sql(f"""
    SELECT MIN(CAST(created_at AS TIMESTAMP)) AS earliest, MAX(CAST(created_at AS TIMESTAMP)) AS latest,
           COUNT(DISTINCT created_at) AS distinct_created_at_values,
           COUNT(DISTINCT updated_at) AS distinct_updated_at_values
    FROM read_csv_auto('{path}')
""").show()

┌─────────────────────┬─────────────────────┬────────────────────────────┬────────────────────────────┐
│      earliest       │       latest        │ distinct_created_at_values │ distinct_updated_at_values │
│      timestamp      │      timestamp      │           int64            │           int64            │
├─────────────────────┼─────────────────────┼────────────────────────────┼────────────────────────────┤
│ 2020-01-01 00:00:00 │ 2020-01-01 00:00:00 │                          1 │                          1 │
└─────────────────────┴─────────────────────┴────────────────────────────┴────────────────────────────┘



In [11]:
# 9. FORMAT CONSISTENCY AUDIT
bad_sku = con.sql(f"""
    SELECT COUNT(*) FROM read_csv_auto('{path}')
    WHERE NOT REGEXP_MATCHES(sku, '^NP-SKU-[0-9]{{5}}$')
""").fetchone()[0]
print(f"SKUs not matching NP-SKU-##### format: {bad_sku}")

con.sql(f"SELECT DISTINCT unit_of_measure FROM read_csv_auto('{path}')").show()
con.sql(f"SELECT DISTINCT category_l1 FROM read_csv_auto('{path}')").show()
con.sql(f"SELECT DISTINCT is_own_brand, is_active FROM read_csv_auto('{path}')").show()

SKUs not matching NP-SKU-##### format: 0
┌─────────────────┐
│ unit_of_measure │
│     varchar     │
├─────────────────┤
│ 100g            │
│ pack            │
│ each            │
│ kg              │
│ litre           │
└─────────────────┘

┌─────────────┐
│ category_l1 │
│   varchar   │
├─────────────┤
│ Household   │
│ Health      │
│ Drink       │
│ Food        │
└─────────────┘

┌──────────────┬───────────┐
│ is_own_brand │ is_active │
│   boolean    │  boolean  │
├──────────────┼───────────┤
│ true         │ true      │
│ false        │ false     │
│ false        │ true      │
└──────────────┴───────────┘



In [35]:
# --- 10. Timezone drift: hour-of-day distribution ---
# created_at
print("\ncreated_at")
print(con.sql(f"""
    SELECT EXTRACT(hour FROM created_at) AS hour, COUNT(*) AS n
    FROM read_csv_auto('{path}') GROUP BY hour ORDER BY hour
""").df())

# updated_at
print("\nupdated_at")
print(con.sql(f"""
    SELECT EXTRACT(hour FROM updated_at) AS hour, COUNT(*) AS n
    FROM read_csv_auto('{path}') GROUP BY hour ORDER BY hour
""").df())


#  Logical time order: nothing should be "updated" before it was "created" ---
print("\n Logical time order: nothing should be 'updated' before it was 'created'")
print(con.sql(f"""
    SELECT COUNT(*) AS backwards_count
    FROM read_csv_auto('{path}') WHERE updated_at < created_at
""").df())


created_at
   hour    n
0     0  150

updated_at
   hour    n
0     0  150

 Logical time order: nothing should be 'updated' before it was 'created'
   backwards_count
0                0


In [34]:
results = []

# CREATED_AT
# 1. Hour-of-day spread (timezone drift check)
hours = con.sql(f"SELECT EXTRACT(hour FROM created_at) AS hour, COUNT(*) AS n FROM read_csv_auto('{path}') GROUP BY hour").df()
results.append({
    'check': 'created_at hour-of-day coverage',
    'result': f"{hours['hour'].nunique()} distinct hours used (range {int(hours['hour'].min())}-{int(hours['hour'].max())})",
    'status': 'OK'
})

# 2. Is updated_at a real timestamp or a constant placeholder?
stats = con.sql(f"""
    SELECT COUNT(*) AS total_rows, COUNT(DISTINCT created_at) AS distinct_dates
    FROM read_csv_auto('{path}')
""").df().iloc[0]

results.append({
    'check': 'created_at distinct values',
    'result': f"{stats['distinct_dates']} distinct value(s) across {stats['total_rows']} rows",
    'status': 'OK' if stats['distinct_dates'] > 1 else '⚠️ FLAG — placeholder, not real'
})

# 3. Gaps in product_id sequence
gaps = con.sql(f"""
    WITH bounds AS (SELECT MIN(product_id) mn, MAX(product_id) mx FROM read_csv_auto('{path}')),
    spine AS (SELECT UNNEST(generate_series(mn, mx)) AS id FROM bounds)
    SELECT COUNT(*) AS n FROM spine s LEFT JOIN read_csv_auto('{path}') c ON s.id = c.product_id WHERE c.product_id IS NULL
""").fetchone()[0]
results.append({
    'check': 'product_id sequence gaps',
    'result': f"{gaps} missing ID(s)",
    'status': 'OK' if gaps == 0 else '⚠️ FLAG'
})

# updated_at
# 1. Hour-of-day spread (timezone drift check)
hours = con.sql(f"SELECT EXTRACT(hour FROM updated_at) AS hour, COUNT(*) AS n FROM read_csv_auto('{path}') GROUP BY hour").df()
results.append({
    'check': 'updated_at hour-of-day coverage',
    'result': f"{hours['hour'].nunique()} distinct hours used (range {int(hours['hour'].min())}-{int(hours['hour'].max())})",
    'status': 'OK'
})

# 2. Is updated_at a real timestamp or a constant placeholder?
stats = con.sql(f"""
    SELECT COUNT(*) AS total_rows, COUNT(DISTINCT updated_at) AS distinct_dates
    FROM read_csv_auto('{path}')
""").df().iloc[0]

results.append({
    'check': 'updated_at distinct values',
    'result': f"{stats['distinct_dates']} distinct value(s) across {stats['total_rows']} rows",
    'status': 'OK' if stats['distinct_dates'] > 1 else '⚠️ FLAG — placeholder, not real'
})

summary = pd.DataFrame(results)
summary

,check,result,status
0,created_at hour-of-day coverage,1 distinct hours used (range 0-0),OK
1,created_at distinct values,1 distinct value(s) across 150 rows,"⚠️ FLAG — placeholder, not real"
2,product_id sequence gaps,0 missing ID(s),OK
3,updated_at hour-of-day coverage,1 distinct hours used (range 0-0),OK
4,updated_at distinct values,1 distinct value(s) across 150 rows,"⚠️ FLAG — placeholder, not real"


In [31]:
# 11. Format consistency: Flag columns that contain mixed data types
def format_consistency_check(con, path):
    # Step 1: get every column name in the file
    schema = con.sql(f"DESCRIBE SELECT * FROM read_csv_auto('{path}')").df()
    columns = schema['column_name'].tolist()

    all_rows = []
    for col in columns:
        # Step 2: for THIS column, classify every value into a bucket
        breakdown = con.sql(f"""
            SELECT
                '{col}' AS column_name,
                CASE
                    WHEN "{col}" IS NULL OR TRIM("{col}") = '' THEN 'NULL/EMPTY'
                    WHEN TRY_CAST("{col}" AS BIGINT) IS NOT NULL THEN 'INTEGER'
                    WHEN TRY_CAST("{col}" AS DOUBLE) IS NOT NULL THEN 'DECIMAL'
                    WHEN TRY_CAST("{col}" AS DATE) IS NOT NULL THEN 'DATE'
                    WHEN TRY_CAST("{col}" AS TIMESTAMP) IS NOT NULL THEN 'TIMESTAMP'
                    WHEN lower("{col}") IN ('true','false') THEN 'BOOLEAN'
                    ELSE 'TEXT'
                END AS detected_type,
                COUNT(*) AS row_count
            FROM read_csv_auto('{path}', all_varchar=True)
            GROUP BY detected_type
            ORDER BY row_count DESC
        """).df()
        all_rows.append(breakdown)

    result = pd.concat(all_rows, ignore_index=True)

    # Step 3: flag a column only if it has 2+ real (non-null) types
    n_types = result[result['detected_type'] != 'NULL/EMPTY'].groupby('column_name')['detected_type'].transform('nunique')
    result['is_mixed'] = (n_types.reindex(result.index).fillna(0) > 1)
    return result

report = format_consistency_check(con, path)
report                       # full breakdown — bare, no print(), so it renders as a table

,column_name,detected_type,row_count,is_mixed
0,product_id,INTEGER,150,False
1,sku,TEXT,150,False
2,product_name,TEXT,150,False
3,brand,TEXT,150,False
4,category_l1,TEXT,150,False
5,category_l2,TEXT,150,False
6,category_l3,TEXT,150,False
7,unit_of_measure,TEXT,150,False
8,pack_size,INTEGER,150,False
9,is_own_brand,BOOLEAN,150,False


In [30]:
# 12. Sentinel/Default value usage:

# Part 1: scan for known/common sentinel patterns
SENTINEL_VALUES = [
    '-1', '0', '999', '9999', '99999', '-999',
    'NA', 'N/A', 'NULL', 'NONE', 'UNKNOWN', 'UNK', 'TBD', '-', '?',
    '1900-01-01', '1970-01-01', '9999-12-31', '0001-01-01'
]

def sentinel_scan(con, path, sentinel_values):
    schema = con.sql(f"DESCRIBE SELECT * FROM read_csv_auto('{path}')").df()
    columns = schema['column_name'].tolist()
    values_sql = ", ".join(f"'{v.upper()}'" for v in sentinel_values)

    all_rows = []
    for col in columns:
        df = con.sql(f"""
            SELECT
                '{col}' AS column_name,
                "{col}" AS value,
                COUNT(*) AS row_count
            FROM read_csv_auto('{path}', all_varchar=True)
            WHERE upper(TRIM("{col}")) IN ({values_sql})
            GROUP BY "{col}"
            ORDER BY row_count DESC
        """).df()
        all_rows.append(df)

    return pd.concat(all_rows, ignore_index=True)

sentinel_report = sentinel_scan(con, path, SENTINEL_VALUES)
sentinel_report


,column_name,value,row_count


# PRODUCTS.XLSX

In [12]:
# ==========================================================
# PRODUCTS.XLSX AUDIT — different pricing model than the csv
# ==========================================================
import pandas as pd

xlsx_path = "../../source_database/seed_data/products.xlsx"
products_xlsx = pd.read_excel(xlsx_path)
products_xlsx.to_csv("products_xlsx_export.csv", index=False)

path = "products_xlsx_export.csv"

In [13]:
# 1. SCHEMA AUDIT
con.sql(f"DESCRIBE SELECT * FROM read_csv_auto('{path}')").show()

┌─────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│   column_name   │ column_type │  null   │   key   │ default │  extra  │
│     varchar     │   varchar   │ varchar │ varchar │ varchar │ varchar │
├─────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ product_id      │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ sku             │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ product_name    │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ brand           │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ category_l1     │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ category_l2     │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ category_l3     │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ unit_of_measure │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ unit_price_gbp  │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ is_active       │ BOOLEAN     │ YES 

In [14]:
con.sql(f"SELECT * FROM read_csv_auto('{path}') LIMIT 5").show()

┌────────────┬──────────────┬──────────────────────┬──────────┬───┬───────────┬─────────────┬────────────┐
│ product_id │     sku      │     product_name     │  brand   │ … │ is_active │ supplier_id │ created_at │
│   int64    │   varchar    │       varchar        │ varchar  │   │  boolean  │    int64    │    date    │
├────────────┼──────────────┼──────────────────────┼──────────┼───┼───────────┼─────────────┼────────────┤
│          1 │ NP-SKU-00001 │ PepsiCo Organic Ve…  │ Nestle   │ … │ true      │          19 │ 2021-03-02 │
│          2 │ NP-SKU-00002 │ PepsiCo Full Fat M…  │ Nestle   │ … │ true      │          16 │ 2020-05-28 │
│          3 │ NP-SKU-00003 │ Danone Mature Ched…  │ Kelloggs │ … │ true      │          14 │ 2020-06-16 │
│          4 │ NP-SKU-00004 │ NorthPeak Own Bran…  │ PepsiCo  │ … │ true      │           1 │ 2018-05-09 │
│          5 │ NP-SKU-00005 │ NorthPeak Own Bran…  │ P&G      │ … │ true      │           3 │ 2019-12-20 │
├────────────┴──────────────┴────────

In [15]:
# 2. COMPLETENESS AUDIT
total_rows = con.sql(f"SELECT COUNT(*) FROM read_csv_auto('{path}')").fetchone()[0]
print(f"Total rows: {total_rows}")

cols = con.sql(f"DESCRIBE SELECT * FROM read_csv_auto('{path}')").df()['column_name'].tolist()
for col in cols:
    null_count = con.sql(f"SELECT COUNT(*) - COUNT({col}) FROM read_csv_auto('{path}')").fetchone()[0]
    pct = round(null_count / total_rows * 100, 2)
    print(f"{col}: {null_count} nulls ({pct}%)")

Total rows: 150
product_id: 0 nulls (0.0%)
sku: 0 nulls (0.0%)
product_name: 0 nulls (0.0%)
brand: 0 nulls (0.0%)
category_l1: 0 nulls (0.0%)
category_l2: 0 nulls (0.0%)
category_l3: 0 nulls (0.0%)
unit_of_measure: 0 nulls (0.0%)
unit_price_gbp: 0 nulls (0.0%)
is_active: 0 nulls (0.0%)
supplier_id: 0 nulls (0.0%)
created_at: 0 nulls (0.0%)


In [16]:
# 3. UNIQUENESS AUDIT
for col in ["product_id", "sku"]:
    dupes = con.sql(f"""
        SELECT {col}, COUNT(*) AS row_count FROM read_csv_auto('{path}')
        GROUP BY {col} HAVING COUNT(*) > 1
    """).df()
    print(f"{col}: {len(dupes)} duplicated values")

product_id: 0 duplicated values
sku: 0 duplicated values


In [17]:
# 4. REFERENTIAL INTEGRITY AUDIT
orphan_suppliers = con.sql(f"""
    SELECT COUNT(*) FROM read_csv_auto('{path}') p
    LEFT JOIN read_csv_auto('../../source_database/seed_data/suppliers.csv') s ON p.supplier_id = s.supplier_id
    WHERE s.supplier_id IS NULL
""").fetchone()[0]
print(f"Orphaned supplier_id rows: {orphan_suppliers}")

Orphaned supplier_id rows: 0


In [18]:
# 5. VALIDITY AUDIT
bad_price = con.sql(f"SELECT COUNT(*) FROM read_csv_auto('{path}') WHERE unit_price_gbp <= 0").fetchone()[0]
print(f"unit_price_gbp <= 0: {bad_price}")

unit_price_gbp <= 0: 0


In [19]:
# 6. BRAND vs PRODUCT_NAME CONSISTENCY CHECK (same test as csv)
mismatch_df = con.sql(f"""
    SELECT COUNT(*) AS mismatches
    FROM read_csv_auto('{path}')
    WHERE NOT REGEXP_MATCHES(LOWER(product_name), LOWER(SPLIT_PART(brand, ' ', 1)))
""").df()
total = con.sql(f"SELECT COUNT(*) FROM read_csv_auto('{path}')").fetchone()[0]
mismatch_count = mismatch_df['mismatches'][0]
print(f"product_name doesn't mention its own brand: {mismatch_count} / {total} ({round(mismatch_count/total*100,1)}%)")

product_name doesn't mention its own brand: 129 / 150 (86.0%)


In [20]:
# 7. DISTRIBUTION / OUTLIER AUDIT
con.sql(f"""
    SELECT MIN(unit_price_gbp) AS min_price, MAX(unit_price_gbp) AS max_price, AVG(unit_price_gbp) AS avg_price
    FROM read_csv_auto('{path}')
""").show()

┌───────────┬───────────┬────────────────────┐
│ min_price │ max_price │     avg_price      │
│  double   │  double   │       double       │
├───────────┼───────────┼────────────────────┤
│      0.52 │     24.94 │ 12.403933333333335 │
└───────────┴───────────┴────────────────────┘



In [21]:
# 8. FRESHNESS AUDIT — note: this file only has created_at, no updated_at
con.sql(f"""
    SELECT MIN(CAST(created_at AS DATE)) AS earliest, MAX(CAST(created_at AS DATE)) AS latest,
           COUNT(DISTINCT created_at) AS distinct_created_at_values
    FROM read_csv_auto('{path}')
""").show()

┌────────────┬────────────┬────────────────────────────┐
│  earliest  │   latest   │ distinct_created_at_values │
│    date    │    date    │           int64            │
├────────────┼────────────┼────────────────────────────┤
│ 2018-01-06 │ 2024-12-11 │                        143 │
└────────────┴────────────┴────────────────────────────┘



In [22]:
# 9. FORMAT CONSISTENCY AUDIT
bad_sku = con.sql(f"""
    SELECT COUNT(*) FROM read_csv_auto('{path}')
    WHERE NOT REGEXP_MATCHES(sku, '^NP-SKU-[0-9]{{5}}$')
""").fetchone()[0]
print(f"SKUs not matching NP-SKU-##### format: {bad_sku}")

con.sql(f"SELECT DISTINCT unit_of_measure FROM read_csv_auto('{path}')").show()
con.sql(f"SELECT DISTINCT category_l1 FROM read_csv_auto('{path}')").show()
con.sql(f"SELECT DISTINCT is_active FROM read_csv_auto('{path}')").show()

SKUs not matching NP-SKU-##### format: 0
┌─────────────────┐
│ unit_of_measure │
│     varchar     │
├─────────────────┤
│ can             │
│ bottle          │
│ pack            │
│ bag             │
│ box             │
│ each            │
│ litre           │
│ kg              │
└─────────────────┘

┌─────────────┐
│ category_l1 │
│   varchar   │
├─────────────┤
│ Food        │
│ Drink       │
│ Household   │
└─────────────┘

┌───────────┐
│ is_active │
│  boolean  │
├───────────┤
│ true      │
└───────────┘



# Comparison

In [23]:
# ==========================================================
# PRODUCTS.CSV vs PRODUCTS.XLSX — FULL COMPARISON AUDIT
# ==========================================================
import pandas as pd

csv_path = "../../source_database/seed_data/products.csv"
xlsx_path = "../../source_database/seed_data/products.xlsx"

csv_df = pd.read_csv(csv_path)
xlsx_df = pd.read_excel(xlsx_path)

print(f"products.csv  -> {csv_df.shape[0]} rows, {csv_df.shape[1]} columns")
print(f"products.xlsx -> {xlsx_df.shape[0]} rows, {xlsx_df.shape[1]} columns")

products.csv  -> 150 rows, 17 columns
products.xlsx -> 150 rows, 12 columns


In [24]:
# STEP 1 — SCHEMA COMPARISON
csv_cols = set(csv_df.columns)
xlsx_cols = set(xlsx_df.columns)

only_in_csv  = csv_cols - xlsx_cols
only_in_xlsx = xlsx_cols - csv_cols
shared_cols  = csv_cols & xlsx_cols

print("Columns only in csv  (different pricing model):", sorted(only_in_csv))
print("Columns only in xlsx (different pricing model):", sorted(only_in_xlsx))
print("Columns in both (same name)                    :", sorted(shared_cols))

Columns only in csv  (different pricing model): ['cost_price', 'is_own_brand', 'pack_size', 'rrp', 'updated_at', 'vat_rate']
Columns only in xlsx (different pricing model): ['unit_price_gbp']
Columns in both (same name)                    : ['brand', 'category_l1', 'category_l2', 'category_l3', 'created_at', 'is_active', 'product_id', 'product_name', 'sku', 'supplier_id', 'unit_of_measure']


In [25]:
# STEP 2 — product_id OVERLAP
csv_ids  = set(csv_df["product_id"])
xlsx_ids = set(xlsx_df["product_id"])
shared_ids = csv_ids & xlsx_ids
print(f"product_id in both files: {len(shared_ids)} out of {len(csv_ids)} (csv) / {len(xlsx_ids)} (xlsx)")

product_id in both files: 150 out of 150 (csv) / 150 (xlsx)


In [26]:
# STEP 3 — FIELD-BY-FIELD COMPARISON ON TRULY SHARED COLUMNS
# product_name, brand, category_l1/l2/l3 etc are the real test:
# does product_id=1 describe the SAME product in both files?
compare_cols = sorted(shared_cols - {"product_id"})

merged = csv_df.merge(xlsx_df, on="product_id", suffixes=("_csv", "_xlsx"))
print(f"Rows compared (product_id present in both files): {len(merged)}")

mismatch_summary = {}
for col in compare_cols:
    mismatches = merged[f"{col}_csv"] != merged[f"{col}_xlsx"]
    mismatch_summary[col] = {
        "mismatches": int(mismatches.sum()),
        "pct": round(mismatches.sum() / len(merged) * 100, 2)
    }

mismatch_df = pd.DataFrame(mismatch_summary).T.sort_values("pct", ascending=False)
mismatch_df

Rows compared (product_id present in both files): 150


,mismatches,pct
created_at,150.0,100.00
product_name,150.0,100.00
category_l3,143.0,95.33
brand,140.0,93.33
supplier_id,140.0,93.33
category_l2,134.0,89.33
unit_of_measure,129.0,86.00
category_l1,93.0,62.00
is_active,8.0,5.33
sku,0.0,0.00


In [27]:
# STEP 4 — SAMPLE EVIDENCE ROWS
sample_mismatch = merged[merged["product_name_csv"] != merged["product_name_xlsx"]][
    ["product_id", "product_name_csv", "product_name_xlsx", "brand_csv", "brand_xlsx"]
].head(10)
sample_mismatch

,product_id,product_name_csv,product_name_xlsx,brand_csv,brand_xlsx
0,1,Kelloggs Fruit 6pk,PepsiCo Organic Veg Mix,Unilever,Nestle
1,2,Warburtons Milk 1kg,PepsiCo Full Fat Milk,Walkers,Nestle
2,3,Walkers Cheese 1kg,Danone Mature Cheddar,Coca-Cola,Kelloggs
3,4,Kelloggs Yoghurt 500g,NorthPeak Own Brand White Loaf,Cadbury,PepsiCo
4,5,Nestle Beef 6pk,NorthPeak Own Brand Croissant 4pk,NorthPeak Own,P&G
5,6,Walkers Chicken 6pk,Coca-Cola Chicken Breast,McVities,NorthPeak Own Brand
6,7,NorthPeak Own Pork 1kg,PepsiCo Beef Mince,Procter & Gamble,Nestle
7,8,McVities Fresh Fish 1kg,Kelloggs Diet Cola 24pk,Unilever,Danone
8,9,Warburtons Frozen Fish 4pk,Reckitt Orange Juice 1L,Warburtons,Unilever
9,10,Walkers Bread 6pk,Nestle Lager 12pk,Unilever,Heinz


In [28]:
# STEP 5 — FINAL SUMMARY
print("=== products.csv vs products.xlsx — comparison summary ===")
print(f"products.csv : {len(csv_df)} rows, {len(csv_df.columns)} columns")
print(f"products.xlsx: {len(xlsx_df)} rows, {len(xlsx_df.columns)} columns")
print(f"Columns only in csv : {sorted(only_in_csv)}")
print(f"Columns only in xlsx: {sorted(only_in_xlsx)}")
print(f"product_id shared between files: {len(shared_ids)}")
print()
print("Field mismatch rate on shared-name columns:")
print(mismatch_df)

=== products.csv vs products.xlsx — comparison summary ===
products.csv : 150 rows, 17 columns
products.xlsx: 150 rows, 12 columns
Columns only in csv : ['cost_price', 'is_own_brand', 'pack_size', 'rrp', 'updated_at', 'vat_rate']
Columns only in xlsx: ['unit_price_gbp']
product_id shared between files: 150

Field mismatch rate on shared-name columns:
                 mismatches     pct
created_at            150.0  100.00
product_name          150.0  100.00
category_l3           143.0   95.33
brand                 140.0   93.33
supplier_id           140.0   93.33
category_l2           134.0   89.33
unit_of_measure       129.0   86.00
category_l1            93.0   62.00
is_active               8.0    5.33
sku                     0.0    0.00
